# Linear Regression from Scratch with PyTorch

In [ ]:
import numpy as np
import torch

from sklearn import compose, datasets, linear_model, metrics, model_selection
from sklearn import pipeline, preprocessing

## Load the California Housing dataset

In [ ]:
housing_dataset = datasets.fetch_california_housing(
    as_frame=True
)

In [ ]:
print(housing_dataset["DESCR"])

In [ ]:
housing_features_df = housing_dataset["data"]
median_house_value = housing_dataset["target"]

In [ ]:
housing_features_df.info()

In [ ]:
_ = median_house_value.hist()

## Train/Val Split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        housing_features_df,
        median_house_value,
        test_size=0.20,
        shuffle=True,
        random_state=RANDOM_STATE
    )
)


In [ ]:
train_features_df.info()

In [ ]:
val_features_df.info()

## Prepare the data

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def dataframe_to_tensor(df, dtype=torch.float32):
    arr = df.to_numpy()
    return array_to_tensor(arr, dtype)


def series_to_tensor(s, dtype=torch.float32):
    df = s.to_frame()
    return dataframe_to_tensor(df, dtype)


prepare_housing_features = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_housing_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor
    )
)



In [ ]:
X_train = prepare_housing_features.fit_transform(train_features_df)
X_val = prepare_housing_features.transform(val_features_df)


In [ ]:
y_train = prepare_housing_target.fit_transform(train_target)
y_val = prepare_housing_target.transform(val_target)


## Linear Regression using Tensors

### Initialize parameters

In [ ]:
prng = torch.manual_seed(42)

_, n_features = X_train.shape

weights = torch.randn((n_features, 1), requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

### Define our model and loss functions

In [ ]:
def model_fn(X):
    return X @ weights + bias


def loss_fn(y_true, y_pred):
    error = y_true - y_pred
    mse = torch.mean(error**2)
    return mse


### Training using full batch gradient descent

In [ ]:
learning_rate = 0.5
n_epochs = 100

for epoch in range(n_epochs):
    # forward pass
    y_pred = model_fn(X_train)
    train_loss = loss_fn(y_train, y_pred)

    # backward pass
    train_loss.backward()

    # gradient descent step
    with torch.no_grad():
        bias -= learning_rate * bias.grad
        weights -= learning_rate * weights.grad
        bias.grad.zero_()
        weights.grad.zero_()

    # evaluate using the validation data
    with torch.no_grad():
        y_pred = model_fn(X_val)
        val_loss = loss_fn(y_val, y_pred)

    print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")



## Linear Regression using the Neural Network API

In [ ]:
from torch import nn, optim

### Define our model and loss functions, and our optimizer

In [ ]:
_ = torch.manual_seed(42)

# linear regression only has a single layer
_, n_features = X_train.shape
housing_model = nn.Linear(
    in_features=n_features,
    out_features=1,
    bias=True,
)

# continue using the mean squared error loss
mse_loss = nn.MSELoss()

# define our optimizer
learning_rate = 1e-2
sgd = optim.SGD(
    housing_model.parameters(),
    lr=learning_rate
)

### Training using full batch gradient descent

In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs
    ):

    for epoch in range(n_epochs):
        # forward pass
        y_pred = model_fn(X_train)
        train_loss = criterion(y_train, y_pred)

        # backward pass
        train_loss.backward()

        # gradient descent step
        optimizer.step()
        optimizer.zero_grad()

        # evaluate using the validation data
        with torch.no_grad():
            y_pred = model_fn(X_val)
            val_loss = criterion(y_val, y_pred)

        print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")


In [ ]:
train(
    housing_model,
    mse_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100
)

### Compare results with Scikit-Learn SGDRegressor

In [ ]:
sgd_regressor = pipeline.make_pipeline(
    prepare_housing_features[:-1],
    linear_model.SGDRegressor(
        loss="squared_error",
        penalty=None,
        random_state=RANDOM_STATE,
    )
)

_ = sgd_regressor.fit(train_features_df, train_target)

In [ ]:
train_prediction = sgd_regressor.predict(train_features_df)
train_mse = metrics.mean_squared_error(
    train_target,
    train_prediction
)


val_prediction = sgd_regressor.predict(val_features_df)
val_mse = metrics.mean_squared_error(
    val_target,
    val_prediction
)

print(f"Training loss: {train_mse: .4f}, Validation loss: {val_mse: .4f}")

## Exercise

Load the diabetes dataset using the code in the cell below. Prepare the data and then train a linear regression model using PyTorch Neural Network API.

In [ ]:
diabetes_dataset = datasets.load_diabetes(
    as_frame=True,
    scaled=False
)

In [ ]:
print(diabetes_dataset["DESCR"])

In [ ]:
diabetes_features_df = diabetes_dataset["data"]
diabetes_progression = diabetes_dataset["target"]

In [ ]:
diabetes_features_df.info()

In [ ]:
diabetes_features_df.describe()

In [ ]:
_ = diabetes_progression.hist()

In [ ]:
# INSERT YOUR CODE HERE!

### Solution

In [ ]:
train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        diabetes_features_df,
        diabetes_progression,
        test_size=0.20,
        shuffle=True,
        random_state=RANDOM_STATE
    )
)


In [ ]:
prepare_diabetes_features = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            preprocessing.OneHotEncoder(
                drop="if_binary",
                sparse_output=False
            ),
            [
                "sex"
            ]
        ),
        n_jobs=-1,
        remainder=preprocessing.StandardScaler(),
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_diabetes_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor
    )
)

In [ ]:
X_train = prepare_diabetes_features.fit_transform(train_features_df)
X_val = prepare_diabetes_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_diabetes_target.fit_transform(train_target)
y_val = prepare_diabetes_target.transform(val_target)


In [ ]:
print(y_train.shape)
print(y_val.shape)

In [ ]:
_ = torch.manual_seed(42)

# linear regression only has a single layer
_, n_features = X_train.shape
diabetes_model = nn.Linear(
    in_features=n_features,
    out_features=1,
    bias=True,
)

# continue using the mean squared error loss
mse_loss = nn.MSELoss()

# define our optimizer
learning_rate = 1e-2
sgd = optim.SGD(
    diabetes_model.parameters(),
    lr=learning_rate
)

# train the model
train(
    diabetes_model,
    mse_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=1000
)

## Exercise (Optional)

Confirm that your results above are similar to those obtained using SGDRegressor from Scikit-Learn.


In [ ]:
# INSERT YOUR CODE HERE!

### Solution

In [ ]:
sgd_regressor = pipeline.make_pipeline(
    prepare_diabetes_features[:-1],
    linear_model.SGDRegressor(
        loss="squared_error",
        penalty=None,
        random_state=RANDOM_STATE,
    )
)

_ = sgd_regressor.fit(train_features_df, train_target)

In [ ]:
train_prediction = sgd_regressor.predict(train_features_df)
train_mse = metrics.mean_squared_error(
    train_target,
    train_prediction
)


val_prediction = sgd_regressor.predict(val_features_df)
val_mse = metrics.mean_squared_error(
    val_target,
    val_prediction
)

print(f"Training loss: {train_mse: .4f}, Validation loss: {val_mse: .4f}")